In [1]:
%reload_ext autoreload

In [2]:
import os
import pandas as pd
import pickle

In [3]:
folder = "remove_partial"
train_set = "train_set.csv"
test_set = "test_set.csv"
calibration_set = "calibration_set.csv"

In [4]:
def load_df(folder,data_name):
    data_path = os.path.join(folder,data_name)
    df = pd.read_csv(data_path)
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time'] = pd.to_datetime(df['end_time'])
    df["start"] = df["concept:name"] + "_start"
    df["start"] = df["start"].apply(lambda row: row.replace(" ", "_").lower())
    df["end"] = df["concept:name"] + "_end"
    df["end"]= df["end"].apply(lambda row: row.replace(" ", "_").lower())
    return df

def store_df(folder,data_name, df):
    data_path = os.path.join(folder,data_name)
    df.to_csv(data_path)
    
def store_file(folder,name,obj):
    file_path = os.path.join(folder,f"{name}.pickle")
    with open(file_path, 'wb') as file:
        pickle.dump(obj, file, protocol=pickle.HIGHEST_PROTOCOL)
    
def from_df_to_trace(df): 
    ccns = df["case:concept:name"].unique()
    return {ccn: from_ccn_to_trace(df,ccn) for ccn in ccns}

def from_ccn_to_trace(df,ccn):
    ddf = df[df["case:concept:name"] == ccn][["start","end","start_time","end_time"]]
    init = ddf.iloc[0]["start_time"]
    ddf["start_time"] = (ddf["start_time"] - init)
    ddf["end_time"] = (ddf["end_time"] - init)
    ddf["start_time"] = ddf["start_time"].apply(lambda data: data.total_seconds())
    ddf["end_time"] = ddf["end_time"].apply(lambda data: data.total_seconds())
    trace = list(ddf[["start_time","start"]].itertuples(index=False,name=None)) + list(ddf[["end_time","end"]].itertuples(index=False,name=None)) 
    trace = [(time,{event}) for (time,event) in trace]
    return sorted(trace, key=lambda event: event[0])

In [5]:
# load dataset
df_train = load_df(folder,train_set)
df_test = load_df(folder,test_set)
df_calibration = load_df(folder,calibration_set)

In [12]:
def compute_max_time_lenght(df):
    df_min = df.groupby("case:concept:name").min()
    df_max =  df.groupby("case:concept:name").max()
    return (df_max["end_time"] - df_min["start_time"]).max()

In [18]:
train_max_length = compute_max_time_lenght(df_train)
test_max_length = compute_max_time_lenght(df_test)
calibration_max_length = compute_max_time_lenght(df_calibration)
max_time_lenght = max(train_max_length,test_max_length,calibration_max_length).total_seconds()
print("Max time_length (sec): ", max_time_lenght )

Max time_length (sec):  4130034.303


In [19]:
from mitl.Semantics import load_formula, evaluate_boolean_semantics, evaluate_robustness_semantics
property1 = load_formula(f"G[0,INF](a_create_application_end --> F[0,{24*3600}]o_sent_end)")
property2 = load_formula(f"G[0,INF](o_sent_end --> F[0,{9*24*3600}] w_call_after_offers_end)")
property3 = load_formula(f"G[0,INF](w_validate_application_start --> F[0,{24*3600}] (w_call_incomplete_files_start | w_validate_application_end ))")

properties = [property1, property2, property3]

trace_train  = from_df_to_trace(df_train)
train_properties_dict = {aid : [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_train.items()}
store_file(folder,"train_prop",train_properties_dict)

trace_test  = from_df_to_trace(df_test)
test_properties_dict = {aid: [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_test.items()}
store_file(folder,"test_prop",test_properties_dict)

trace_calibration  = from_df_to_trace(df_calibration)
calibration_properties_dict = { aid: [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_calibration.items()}
store_file(folder,"calibration_prop",calibration_properties_dict)

In [7]:
############################################# END ############################
# (devo capire se rimovere o modificare la parte sotto)

In [86]:
# df_train[df_train["case:concept:name"] == "Application_1029342308"]
# df_train_properties[df_train_properties["Propoerty 2"] == -float("inf")]

In [70]:
df_train_properties = pd.DataFrame(train_properties, columns = ['case:concept:name','Propoerty 1', 'Property 2', 'Proprty 3']) 
df_test_properties = pd.DataFrame(test_properties, columns = ['case:concept:name','Property 1', 'Property 2', 'Property 3']) 
df_calibration_properties = pd.DataFrame(calibration_properties, columns = ['case:concept:name','Property 1', 'Property 2', 'Property 3']) 

In [7]:
from mitl.Semantics import load_formula, evaluate_boolean_semantics, evaluate_robustness_semantics

train_properties = [[aid,]+[evaluate_boolean_semantics(trace,prop) for prop in properties] + [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_train.items()]
test_properties = [[aid,]+[evaluate_boolean_semantics(trace,prop) for prop in properties] + [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_test.items()]
calibration_properties = [[aid,]+[evaluate_boolean_semantics(trace,prop) for prop in properties] + [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_calibration.items()]

In [8]:
for res in [train_properties, test_properties, calibration_properties]:
    print("check: ", sum([r[1] == (r[4]>0) for r in res]) / len(res), sum([r[2] == (r[5]>0) for r in res]) / len(res), sum([r[3] == (r[6]>0) for r in res]) / len(res))
    print("balance: ", sum([r[1] for r in res])/len(res),sum([r[2] for r in res])/len(res),sum([r[3] for r in res])/len(res))
    print("---")

check:  1.0 1.0 1.0
balance:  0.484 0.492 0.523
---
check:  1.0 1.0 1.0
balance:  0.471 0.551 0.521
---
check:  1.0 1.0 1.0
balance:  0.454 0.531 0.635
---


In [20]:
res = calibration_properties_dict
for res in [train_properties_dict,test_properties_dict,calibration_properties_dict]:
    print(sum([a[1]>0 for a in res.values()])/len(res))

0.5077706842071322
0.5414812876210416
0.5499026290165531


In [12]:
len(calibration_properties_dict)

4108

In [20]:
trace_train

{'Application_1000158214': [(0.0, {'a_create_application_start'}),
  (0.0, {'a_create_application_end'}),
  (0.041, {'a_submitted_start'}),
  (0.041, {'a_submitted_end'}),
  (0.383, {'w_handle_leads_start'}),
  (69.908, {'w_handle_leads_end'}),
  (69.92, {'w_complete_application_start'}),
  (69.929, {'a_concept_start'}),
  (69.929, {'a_concept_end'}),
  (338140.85, {'a_accepted_start'}),
  (338140.85, {'a_accepted_end'}),
  (338255.558, {'o_create_offer_start'}),
  (338255.558, {'o_create_offer_end'}),
  (338256.813, {'o_created_start'}),
  (338256.813, {'o_created_end'}),
  (338539.991, {'o_sent_start'}),
  (338539.991, {'o_sent_end'}),
  (338540.025, {'w_complete_application_end'}),
  (338540.044, {'w_call_after_offers_start'}),
  (338540.048, {'a_complete_start'}),
  (338540.048, {'a_complete_end'}),
  (603608.775, {'w_call_after_offers_end'}),
  (603608.782, {'w_validate_application_start'}),
  (603610.026, {'a_validating_start'}),
  (603610.026, {'a_validating_end'}),
  (603646.80

In [21]:
train_properties_dict

{'Application_1000158214': [-252139.99099999998,
  265068.78400000004,
  -4045.655999999959],
 'Application_1000311556': [85252.148, -1863903.4360000002, inf],
 'Application_1000339879': [15916.990000000005,
  346549.841,
  -357587.1009999999],
 'Application_100034150': [65712.549, 274477.011, -269635.478],
 'Application_1000474975': [85834.738, 102632.83699999994, -345041.2039999999],
 'Application_1000557783': [85964.272, 35568.58799999999, -272795.9019999999],
 'Application_1000647090': [-9465.525999999998,
  -1202730.9020000002,
  -443075.8530000001],
 'Application_1000671285': [9970.871, 264944.049, -12560.085999999894],
 'Application_1000691650': [-941452.793, -1895786.6069999998, inf],
 'Application_1000806256': [80455.344,
  -428243.90299999993,
  -92194.20999999996],
 'Application_1000867366': [-42796.393, 240781.7420000001, 9549.69299999997],
 'Application_1001177986': [-64573.22399999999,
  262918.6819999999,
  -507777.05000000005],
 'Application_1001274919': [-23949.7449999

In [23]:
evaluate_robustness_semantics(trace,property3)

NameError: name 'trace' is not defined

In [24]:
evaluate_robustness_semantics(trace_train["Application_1000691650"],property3)

inf

In [25]:
trace_train["Application_1000691650"]

[(0.0, {'a_create_application_start'}),
 (0.0, {'a_create_application_end'}),
 (0.055, {'a_submitted_start'}),
 (0.055, {'a_submitted_end'}),
 (0.232, {'w_handle_leads_start'}),
 (31.306, {'w_handle_leads_end'}),
 (31.314, {'w_complete_application_start'}),
 (31.32, {'a_concept_start'}),
 (31.32, {'a_concept_end'}),
 (100425.391, {'a_accepted_start'}),
 (100425.391, {'a_accepted_end'}),
 (100570.302, {'o_create_offer_start'}),
 (100571.602, {'o_created_start'}),
 (100616.997, {'o_sent_start'}),
 (100617.029, {'w_complete_application_end'}),
 (100617.049, {'w_call_after_offers_start'}),
 (100617.062, {'a_complete_start'}),
 (100617.062, {'a_complete_end'}),
 (1027833.782, {'o_create_offer_end'}),
 (1027834.962, {'o_created_end'}),
 (1027852.793, {'o_sent_end'}),
 (3701239.315, {'a_cancelled_start'}),
 (3701239.315, {'a_cancelled_end'}),
 (3701239.363, {'o_cancelled_start'}),
 (3701239.389, {'o_cancelled_end'}),
 (3701239.4, {'w_call_after_offers_end'})]